# OPT-01 Introduction to edges workflow and API

This notebook introduces the basic `edges` workflow through two small JSON methods: one with direct matching only, and one with location-specific characterization factors.


## Learning goals

After this notebook, you should be able to:

- inspect bundled `edges` JSON method files
- run a simple `EdgeLCIA` workflow with direct exchange matching
- understand how location constraints change the mapping process
- use the main helper functions for aggregate, dynamic, contained, and global fallback mappings
- inspect the exchange-level CFs actually applied in a calculation


## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7
- Brightway documentation: https://docs.brightway.dev/
- Bundled `edges` JSON examples in `tutorials/OPTIONAL/assets/edges_examples/`


## 1) Locate the JSON examples used in this notebook

These two files are bundled with the course repository and are used directly below:

- `lcia_example_1.json`: direct exchange matching
- `lcia_example_2.json`: location-specific characterization with spatial helper mappings

The workflow cells below assume a Brightway project named `ecoinvent-3.10.1-cutoff`.


In [ ]:
import json
from pathlib import Path

import bw2data as bd
import pandas as pd
from IPython.display import display
from edges import EdgeLCIA, get_available_methods

example_1_path = Path('assets/edges_examples/lcia_example_1.json')
example_2_path = Path('assets/edges_examples/lcia_example_2.json')

pd.DataFrame([
    {
        'json file': example_1_path.name,
        'full path': str(example_1_path),
        'main idea': 'Direct exchange matching',
    },
    {
        'json file': example_2_path.name,
        'full path': str(example_2_path),
        'main idea': 'Regionalized matching with consumer locations',
    },
])


## 2) Example 1: a simple exchange-specific method

This first JSON method has only three rules:

- `Carbon dioxide` gets a CF of `1.0`
- `Methane, fossil` gets a CF of `28.0`
- `Dinitrogen monoxide` gets a CF of `265.0`

No location condition is used yet. The main lesson is the minimal `EdgeLCIA` workflow.


In [ ]:
example_1_data = json.loads(example_1_path.read_text())
print(json.dumps(example_1_data, indent=2))

The next cell selects one activity from the `bafu` database.

In [ ]:
bd.projects.set_current("paris-lca-course-2026")
db = bd.Database("bafu")
activity = db.random()
activity

We now initiate the `EdgeLCIA` class like so:

In [ ]:
lca_1 = EdgeLCIA(
    demand={activity: 1},
    method=('some', 'method'), # we give it some name
    filepath=str(example_1_path), # we point here to the JSON file
)

In [ ]:
lca_1.lci() # we solve the system, as `bw2calc` usually does
lca_1.map_exchanges() # we map characterization factors with eligible edges
lca_1.evaluate_cfs() # we evlalute the factors
lca_1.lcia() # we compute H
print('Example 1 score:', lca_1.score)


We can print some statistics about the mapping of exchanges

In [ ]:
lca_1.statistics()

And we can print a nice dataframe listing characterized (and uncharacterized) exchanges.

In [ ]:
cf_table_1 = lca_1.generate_cf_table()
print(len(cf_table_1))
cf_table_1.head()

In [ ]:
# we can also print all of the exchange, including those that did not get a CF
# just to check things
cf_table_1 = lca_1.generate_cf_table(include_unmatched=True)
print(len(cf_table_1))
cf_table_1.head()

In [ ]:
(
    cf_table_1.groupby('supplier name')['CF']
    .mean()
    .sort_values(ascending=False)
    .to_frame('Mean CF')
)

Wait, there's an issue here: isn't `Carbon dioxide, in air` supposed to represent CO2 uptake? How do we fix this?

## 3) Example 2: add consumer locations

This second JSON method still targets `Carbon dioxide`, but now the applied CF depends on the consumer location:

- `CH` gets `1.0`
- `FR` gets `2.0`
- `RER` gets `3.0`

So the workflow starts the same way, but direct matching alone will only cover the exchanges whose locations already match those rules.


In [ ]:
example_2_data = json.loads(example_2_path.read_text())
print(json.dumps(example_2_data, indent=2))


In [ ]:
lca_2_direct = EdgeLCIA(
    demand={activity: 1},
    method=('some', 'method'),
    filepath=str(example_2_path),
)

In [ ]:
lca_2_direct.lci()
lca_2_direct.map_exchanges()
lca_2_direct.evaluate_cfs()
lca_2_direct.lcia()
print('Example 2 score:', lca_2_direct.score)

In [ ]:
cf_table_2_direct = lca_2_direct.generate_cf_table()
cf_table_2_direct.head()

In [ ]:
(
    cf_table_2_direct.groupby('consumer location')['CF']
    .mean()
    .sort_values(ascending=False)
    .to_frame('Mean CF')
)

## 4) Extend Example 2 with the spatial helper functions

We now add the helper functions that expand the geographic coverage:

- `map_aggregate_locations()` for aggregate regions such as `RER`
- `map_dynamic_locations()` for dynamic regions such as `RoW`
- `map_contained_locations()` for subregions contained in larger regions
- `map_remaining_locations_to_global()` for the final global fallback

This is the first full `edges` workflow that we will reuse later in Day 2.


In [ ]:
lca_3 = EdgeLCIA(
    demand={activity: 1},
    method=('some', 'method'),
    filepath=str(example_2_path),
)

In [ ]:
lca_3.lci()
lca_3.map_exchanges() # direct matches
lca_3.map_aggregate_locations() # handles aggregate locations
lca_3.map_dynamic_locations() # handles dynamic locations
lca_3.map_contained_locations() # handles contained locations
lca_3.map_remaining_locations_to_global() # handles the rest
lca_3.evaluate_cfs()
lca_3.lcia()
print('Example 2 score after helper mappings:', lca_3.score)

In [ ]:
lca_3.statistics()

In [ ]:
cf_table_3 = lca_3.generate_cf_table()
cf_table_3.head()

In [ ]:
import matplotlib.pyplot as plt

(
    cf_table_3.groupby("consumer location")["CF"]
    .mean()
    .sort_values(ascending=False)
    .plot.bar(figsize=(10, 6))
)

plt.ylabel("Mean CF after helper mappings")
plt.xlabel("Consumer location")
plt.title("Mean CF by consumer location")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 5) Preview the built-in method library

Besides ad hoc JSON files, `edges` also ships pre-generated method definitions.


In [ ]:
available_methods = get_available_methods()
print('Number of available methods:', len(available_methods))

for method in available_methods:
    print(method)


## Checkpoint 1

Answer these three questions:

- Which call is enough for Example 1, but insufficient for the regionalized Example 2?
- Which helper function handles dynamic locations such as `RoW`?
- Why can weighted-average CFs appear after the helper-mapping workflow?


In [ ]:
# TODO
# answer_1 = ...
# answer_2 = ...
# answer_3 = ...


In [ ]:
print('1. Example 1 only needs `map_exchanges()` because its rules depend on flow identity only, not on geography.')
print('2. Dynamic locations such as `RoW` are handled by `map_dynamic_locations()`.')
print('3. Weighted-average CFs appear because `edges` expands aggregate or dynamic geographies from the available location-specific factors, using default weights such as population unless other weights are provided.')


## Recap

After this notebook, you should now be able to:

- connect the local `edges` example notebooks to their JSON method files
- run the minimal `EdgeLCIA` workflow from `lci()` to `lcia()`
- explain why regionalized methods need more than direct exchange matching
- identify the main helper functions used to broaden spatial coverage
- inspect the exchange-level CFs and weights used by `edges`
